In [8]:
import importlib
from datetime import datetime

from vnpy.trader.constant import Interval
from vnpy_portfoliostrategy.backtesting import BacktestingEngine

# 如果策略代码有修改，需要reload
from vnpy_portfoliostrategy.my_strategies import etf_momentum_strategy
importlib.reload(etf_momentum_strategy)
from vnpy_portfoliostrategy.my_strategies.etf_momentum_strategy import ETFMomentumStrategy

# ETF组合
SYMBOLS = [
    # 股票（3只）
    "510300.SSE",   # 沪深300
    "510500.SSE",   # 中证500
    "159915.SZSE",  # 创业板
    
    # 债券（2只）
    "511010.SSE",   # 国债ETF
    "511260.SSE",   # 10年国债
    
    # 黄金和海外（2只）
    "518880.SSE",   # 黄金ETF
    "513100.SSE",   # 纳指ETF
]

def get_default_config(symbols):
    config = {"rates": {}, "slippages": {}, "sizes": {}, "priceticks": {}}
    for symbol in symbols:
        config["rates"][symbol] = 0.0002      # 手续费万二
        config["slippages"][symbol] = 0.001   # 滑点千分之一
        config["sizes"][symbol] = 1           # ETF一份就是1股
        config["priceticks"][symbol] = 0.001  # 最小价格变动
    return config

engine = BacktestingEngine()
config = get_default_config(SYMBOLS)
initial_capital = 1_000_000

engine.set_parameters(
    vt_symbols=SYMBOLS,
    interval=Interval.DAILY,
    start=datetime(2019, 5, 31),      # 30年国债ETF 2023年4月上市
    end=datetime(2026, 5, 29),
    rates=config["rates"],
    slippages=config["slippages"],
    sizes=config["sizes"],
    priceticks=config["priceticks"],
    capital=initial_capital,
)

setting = {
    # ===== 动量参数 =====
    "m_days": 25,                # 默认动量参考天数
    "auto_day": True,            # 启用ATR动态回溯期
    "min_days": 20,              # 最小回溯期（高波动时）
    "max_days": 60,              # 最大回溯期（低波动时）
    "annual_days": 250,          # 年化交易日数
    
    # ===== 风控参数 =====
    "drop_threshold": 0.95,      # 跌幅阈值（5%）
    "score_min": 0.0,            # 得分下限
    "score_max": 6.0,            # 得分上限
    
    # ===== 仓位参数 =====
    "target_num": 1,             # 目标持仓ETF数量
    "position_pct": 1.0,         # 仓位比例（100%）
    "price_add": 0.01,           # 下单价格偏移
    
    # ===== 交易成本 =====
    "commission_rate": 0.0002,   # 佣金费率（双边万二）
    "min_commission": 1.0,       # 最低佣金
    
    # ===== 初始资金 =====
    "initial_capital": initial_capital,
}

# 运行回测
engine.add_strategy(ETFMomentumStrategy, setting)
engine.load_data()
engine.run_backtesting()
df = engine.calculate_result()
engine.calculate_statistics()
engine.show_chart()

2026-07-03 14:01:21.593289	Start loading historical data
2026-07-03 14:01:21.595914	510300.SSE Historical data loading completed, data count: 1696
2026-07-03 14:01:21.598358	510500.SSE Historical data loading completed, data count: 1696
2026-07-03 14:01:21.599197	159915.SZSE Historical data loading completed, data count: 1695
2026-07-03 14:01:21.599776	511010.SSE Historical data loading completed, data count: 1696
2026-07-03 14:01:21.600258	511260.SSE Historical data loading completed, data count: 1696
2026-07-03 14:01:21.600838	518880.SSE Historical data loading completed, data count: 1696
2026-07-03 14:01:21.601440	513100.SSE Historical data loading completed, data count: 1695
2026-07-03 14:01:21.601455	All historical data has been loaded
2026-07-03 14:01:21.606233	Strategy initialization complete
2026-07-03 14:01:21.606327	Start replaying historical data
2026-07-03 14:01:25.509310	Historical backtest complete
2026-07-03 14:01:25.509464	Start calculating daily mark-to-market profit a

In [7]:
print("\n" + "="*30 + " 策略日志输出 " + "="*30)
for log in engine.logs:
    print(log)  # 每条日志会包含时间和内容


============================== 策略日志输出 ==============================
1970-01-01 00:00:00	ETF动量轮动策略初始化完成
1970-01-01 00:00:00	ETF数量: 7
1970-01-01 00:00:00	目标持仓数: 1
1970-01-01 00:00:00	动态回溯期: 启用
1970-01-01 00:00:00	  回溯期范围: 20 ~ 60天
1970-01-01 00:00:00	策略初始化，初始资金: 1,000,000
2019-11-08 00:00:00+08:00	策略启动
2019-11-11 00:00:00+08:00	【打分排名】Top1: 513100.SSE 得分: 0.5147 | 年化收益: 61.62% | R²: 0.8353 | 回溯期: 30天
2019-11-11 00:00:00+08:00	【调仓执行】买入 513100.SSE (335795手)
2019-11-11 00:00:00+08:00	总权益: 1,000,000 | 持仓数: 1个ETF | 日期: 2019-11-11
2019-11-12 00:00:00+08:00	【调仓执行】调整 513100.SSE: 335795手 -> 336065手
2019-11-12 00:00:00+08:00	总权益: 1,005,845 | 持仓数: 1个ETF | 日期: 2019-11-12
2019-11-22 00:00:00+08:00	【打分排名】Top1: 513100.SSE 得分: 0.5752 | 年化收益: 63.95% | R²: 0.8995 | 回溯期: 28天
2019-11-27 00:00:00+08:00	【打分排名】Top1: 513100.SSE 得分: 0.6112 | 年化收益: 69.01% | R²: 0.8857 | 回溯期: 24天
2019-12-02 00:00:00+08:00	【打分排名】Top1: 513100.SSE 得分: 0.7303 | 年化收益: 80.96% | R²: 0.9020 | 回溯期: 24天
2019-12-12 00:00:00+08:00	【打分排名】Top1

In [ ]:
if engine.trades:
    for trade in engine.trades.values():
        # 只打印 TradeData 实际有的字段
        print(f"时间: {trade.datetime} | 标的: {trade.symbol} | "
              f"方向: {trade.direction.value} | 开平: {trade.offset.value} | "
              f"价格: {trade.price:.3f} | 数量: {trade.volume}")
else:
    print("无成交记录")

时间: 2019-11-12 00:00:00+08:00 | 标的: 513100 | 方向: Long | 开平: Open | 价格: 2.975 | 数量: 335795
时间: 2019-11-13 00:00:00+08:00 | 标的: 513100 | 方向: Long | 开平: Open | 价格: 2.993 | 数量: 270
时间: 2019-12-13 00:00:00+08:00 | 标的: 510500 | 方向: Long | 开平: Open | 价格: 1.531 | 数量: 681893
时间: 2019-12-13 00:00:00+08:00 | 标的: 513100 | 方向: Short | 开平: Close | 价格: 3.108 | 数量: 336065
时间: 2019-12-16 00:00:00+08:00 | 标的: 510500 | 方向: Long | 开平: Open | 价格: 1.542 | 数量: 61
时间: 2019-12-26 00:00:00+08:00 | 标的: 510500 | 方向: Short | 开平: Close | 价格: 1.570 | 数量: 681954
时间: 2019-12-26 00:00:00+08:00 | 标的: 159915 | 方向: Long | 开平: Open | 价格: 1.716 | 数量: 623857
时间: 2019-12-27 00:00:00+08:00 | 标的: 159915 | 方向: Short | 开平: Close | 价格: 1.728 | 数量: 300
时间: 2019-12-30 00:00:00+08:00 | 标的: 510500 | 方向: Long | 开平: Open | 价格: 1.568 | 数量: 674050
时间: 2019-12-30 00:00:00+08:00 | 标的: 159915 | 方向: Short | 开平: Close | 价格: 1.695 | 数量: 623557
时间: 2019-12-31 00:00:00+08:00 | 标的: 510500 | 方向: Short | 开平: Close | 价格: 1.590 | 数量: 135
时间: 2020-01-0

In [6]:
# df.to_csv("rotation_strategy_df.csv", encoding="utf-8-sig")

In [2]:
"""
ETF动量轮动策略回测脚本（含按年统计）
对应策略: ETFMomentumStrategy
"""

import importlib
from datetime import datetime
import pandas as pd
import os

from vnpy.trader.constant import Interval
from vnpy_portfoliostrategy.backtesting import BacktestingEngine

# 如果策略代码有修改，需要reload
from vnpy_portfoliostrategy.my_strategies import etf_momentum_strategy
importlib.reload(etf_momentum_strategy)
from vnpy_portfoliostrategy.my_strategies.etf_momentum_strategy import ETFMomentumStrategy

# ===== 回测标的池 =====
SYMBOLS = [
    # 股票（3只）
    "510300.SSE",   # 沪深300
    "510500.SSE",   # 中证500
    "159915.SZSE",  # 创业板
    
    # 债券（2只）
    "511010.SSE",   # 国债ETF
    "511260.SSE",   # 10年国债
    
    # 黄金和海外（2只）
    "518880.SSE",   # 黄金ETF
    "513100.SSE",   # 纳指ETF
]


def get_default_config(symbols):
    config = {"rates": {}, "slippages": {}, "sizes": {}, "priceticks": {}}
    for symbol in symbols:
        config["rates"][symbol] = 0.0002      # 手续费万二
        config["slippages"][symbol] = 0.001   # 滑点千分之一
        config["sizes"][symbol] = 1           # ETF一份就是1股
        config["priceticks"][symbol] = 0.001  # 最小价格变动
    return config


# ===== 初始化回测引擎 =====
engine = BacktestingEngine()
config = get_default_config(SYMBOLS)
initial_capital = 1_000_000

engine.set_parameters(
    vt_symbols=SYMBOLS,
    interval=Interval.DAILY,
    start=datetime(2015, 1, 1),           # 提前到2015年，积累足够数据
    end=datetime(2026, 5, 29),
    rates=config["rates"],
    slippages=config["slippages"],
    sizes=config["sizes"],
    priceticks=config["priceticks"],
    capital=initial_capital,
)


# ===== 策略参数配置 =====
setting = {
    # ===== 动量参数 =====
    "m_days": 25,
    "auto_day": True,
    "min_days": 20,
    "max_days": 60,
    "annual_days": 250,
    
    # ===== 风控参数 =====
    "drop_threshold": 0.95,
    "score_min": 0.0,
    "score_max": 6.0,
    
    # ===== 仓位参数 =====
    "target_num": 1,
    "position_pct": 1.0,
    "price_add": 0.01,
    
    # ===== 交易成本 =====
    "commission_rate": 0.0002,
    "min_commission": 1.0,
    
    # ===== 初始资金 =====
    "initial_capital": initial_capital,
}


# ===== 运行回测 =====
engine.add_strategy(ETFMomentumStrategy, setting)
engine.load_data()
engine.run_backtesting()

# ===== 获取每日数据 =====
engine.calculate_result()
daily_df = engine.calculate_result()

# 如果daily_df为空或没有数据，手动构建
if daily_df is None or len(daily_df) == 0:
    if hasattr(engine, 'daily_pnl'):
        daily_df = pd.DataFrame({
            'datetime': pd.Series(engine.daily_dt, dtype='datetime64'),
            'net_pnl': engine.daily_pnl,
            'balance': engine.daily_balance,
        })
        daily_df.set_index('datetime', inplace=True)
    else:
        daily_df = engine.calculate_result()

# 确保索引是日期格式
if not isinstance(daily_df.index, pd.DatetimeIndex):
    daily_df.index = pd.to_datetime(daily_df.index)
daily_df = daily_df.sort_index()

# 计算净值（使用balance列）
if 'balance' in daily_df.columns:
    daily_df['nav'] = daily_df['balance']
else:
    # 如果没有balance，从net_pnl计算
    daily_df['nav'] = initial_capital + daily_df['net_pnl'].cumsum()

# ===== 找到策略真正开始交易的日期（跳过预热期） =====
strategy_start_date = None

# 查找第一次有成交的日期
if 'trade_count' in daily_df.columns:
    first_trade_df = daily_df[daily_df['trade_count'] > 0]
    if len(first_trade_df) > 0:
        strategy_start_date = first_trade_df.index[0]

# 如果没有成交，查找净值第一次变化的日期
if strategy_start_date is None:
    initial_nav = daily_df['nav'].iloc[0]
    nav_changed_df = daily_df[abs(daily_df['nav'] - initial_nav) > 1]
    if len(nav_changed_df) > 0:
        strategy_start_date = nav_changed_df.index[0]

# 如果还是找不到，从第60天开始（确保ATR等指标已就绪）
if strategy_start_date is None:
    if len(daily_df) > 60:
        strategy_start_date = daily_df.index[60]
    else:
        strategy_start_date = daily_df.index[0]

print(f"\n策略预热完成，正式开始交易日期: {strategy_start_date.strftime('%Y-%m-%d')}")

# 从策略真正开始日期开始，过滤掉预热期
daily_df = daily_df[daily_df.index >= strategy_start_date]

# ========== 按年统计 ==========
print("\n" + "="*80)
print("📊 按年统计结果")
print("="*80)

# 添加年份列
daily_df['year'] = daily_df.index.year

annual_results = []

for year, group in daily_df.groupby('year'):
    if len(group) == 0:
        continue
    
    # 获取该年净值序列
    nav = group['nav']
    start_nav = nav.iloc[0]
    end_nav = nav.iloc[-1]
    
    # 起始日期和结束日期
    start_date = group.index[0].strftime('%Y-%m-%d')
    end_date = group.index[-1].strftime('%Y-%m-%d')
    
    # 总收益率
    total_return = (end_nav - start_nav) / start_nav
    
    # 年化收益（基于实际交易日数）
    trading_days = len(group)
    if trading_days > 0 and total_return > -1:
        annual_return = (1 + total_return) ** (252 / trading_days) - 1
    else:
        annual_return = 0
    
    # 最大回撤（使用全局累计最高点）
    cummax = nav.cummax()
    drawdown = (nav - cummax) / cummax
    max_drawdown = drawdown.min()
    
    # 夏普比率
    daily_returns = nav.pct_change().dropna()
    if len(daily_returns) > 0 and daily_returns.std() > 0:
        sharpe = (daily_returns.mean() * 252) / (daily_returns.std() * (252 ** 0.5))
    else:
        sharpe = 0
    
    # 成交统计
    total_trades = group['trade_count'].sum() if 'trade_count' in group.columns else 0
    
    annual_results.append({
        '年份': year,
        '起始日期': start_date,
        '结束日期': end_date,
        '年化收益': annual_return,
        '总收益': total_return,
        '最大回撤': max_drawdown,
        '夏普比率': sharpe,
        '起始净值': start_nav,
        '结束净值': end_nav,
        '交易日数': trading_days,
        '成交笔数': total_trades,
    })

# 创建显示用的DataFrame
display_df = pd.DataFrame([{
    '年份': r['年份'],
    '起始日期': r['起始日期'],
    '结束日期': r['结束日期'],
    '年化收益': f"{r['年化收益']:.2%}",
    '总收益': f"{r['总收益']:.2%}",
    '最大回撤': f"{r['最大回撤']:.2%}",
    '夏普比率': f"{r['夏普比率']:.2f}",
    '起始净值': f"{r['起始净值']:,.0f}",
    '结束净值': f"{r['结束净值']:,.0f}",
    '交易日数': r['交易日数'],
    '成交笔数': r['成交笔数']
} for r in annual_results])

print(display_df.to_string(index=False))

# ========== 整体统计 ==========
print(f"\n{'='*80}")
print("📈 整体统计")
print(f"{'='*80}")

if len(annual_results) > 0:
    # 整体收益（从策略开始到结束）
    total_nav_start = annual_results[0]['起始净值']
    total_nav_end = annual_results[-1]['结束净值']
    total_return_all = (total_nav_end - total_nav_start) / total_nav_start
    
    # 整体年化收益
    total_days = sum(r['交易日数'] for r in annual_results)
    if total_days > 0 and total_return_all > -1:
        total_annual_return = (1 + total_return_all) ** (252 / total_days) - 1
    else:
        total_annual_return = 0
    
    # 最大回撤（所有年份中最大的）
    max_drawdown_all = min(r['最大回撤'] for r in annual_results)
    
    # 平均夏普
    avg_sharpe = sum(r['夏普比率'] for r in annual_results) / len(annual_results)
    
    # 正收益年份
    positive_years = [r for r in annual_results if r['总收益'] > 0]
    
    print(f"策略开始日期: {annual_results[0]['起始日期']}")
    print(f"策略结束日期: {annual_results[-1]['结束日期']}")
    print(f"整体总收益: {total_return_all:.2%}")
    print(f"整体年化收益: {total_annual_return:.2%}")
    print(f"最大回撤: {max_drawdown_all:.2%}")
    print(f"平均夏普比率: {avg_sharpe:.2f}")
    print(f"正收益年份: {len(positive_years)}/{len(annual_results)} ({len(positive_years)/len(annual_results):.0%})")
    print(f"总成交笔数: {sum(r['成交笔数'] for r in annual_results)}")

# ===== 保存到CSV =====
desktop = os.path.join(os.path.expanduser('~'), 'Desktop')
file_path = os.path.join(desktop, 'ETFMomentum_backtest_results.csv')

save_df = pd.DataFrame([{
    '年份': r['年份'],
    '起始日期': r['起始日期'],
    '结束日期': r['结束日期'],
    '年化收益_%': f"{r['年化收益']:.2%}",
    '总收益_%': f"{r['总收益']:.2%}",
    '最大回撤_%': f"{r['最大回撤']:.2%}",
    '夏普比率': f"{r['夏普比率']:.2f}",
    '起始净值': r['起始净值'],
    '结束净值': r['结束净值'],
    '交易日数': r['交易日数'],
    '成交笔数': r['成交笔数']
} for r in annual_results])

save_df.to_csv(file_path, index=False, encoding='utf-8-sig')
print(f"\n✅ 结果已保存到: {file_path}")

# ===== 显示图表 =====
engine.show_chart()

2026-07-03 13:42:34.201095	Start loading historical data
2026-07-03 13:42:34.251181	510300.SSE Historical data loading completed, data count: 2769
2026-07-03 13:42:34.285832	510500.SSE Historical data loading completed, data count: 2767
2026-07-03 13:42:34.318340	159915.SZSE Historical data loading completed, data count: 2768
2026-07-03 13:42:34.516515	511010.SSE Historical data loading completed, data count: 2769
2026-07-03 13:42:34.546173	511260.SSE Historical data loading completed, data count: 2123
2026-07-03 13:42:34.580594	518880.SSE Historical data loading completed, data count: 2769
2026-07-03 13:42:34.615461	513100.SSE Historical data loading completed, data count: 2768
2026-07-03 13:42:34.615485	All historical data has been loaded
2026-07-03 13:42:34.619521	Strategy initialization complete
2026-07-03 13:42:34.619559	Start replaying historical data
2026-07-03 13:42:38.552051	Historical backtest complete
2026-07-03 13:42:38.552175	Start calculating daily mark-to-market profit a

KeyError: 'balance'